# End-to-End Project: River Encroachment Analysis

## Real-World Scenario

A water management authority needs to identify buildings within **6 meters of a river/creek** that are at risk during flood events. This end-to-end project demonstrates:

1. **Data Integration**: Combining raster imagery with vector data
2. **Coordinate Reference System (CRS) Management**: Ensuring all datasets align
3. **Spatial Operations**: Creating buffer zones around waterways
4. **Machine Learning**: Automatically detecting buildings from satellite imagery
5. **Spatial Analysis**: Finding buildings that intersect with flood risk zones
6. **Visualization**: Creating interactive maps for decision-makers

This project synthesizes concepts from all previous notebooks:
- **Notebook 01**: CRS, vector/raster data, spatial operations, spatial joins
- **Notebook 02**: Visualization of raster, vector, and combined data
- **Notebook 03**: Semantic segmentation for building detection

## 1. Setup and Data Loading

Import all required libraries and load the datasets from previous notebooks.

In [1]:
# Optional: Install packages if needed
# %pip install leafmap geopandas rasterio geoai-py

import leafmap
import geopandas as gpd
import rasterio
import geoai
import numpy as np
from pathlib import Path
from shapely.geometry import LineString, Polygon
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Define data paths (using data from notebook 03)
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

# Paths to existing data from notebook 03
train_raster_path = data_dir / "naip_rgb_train.tif"
train_vector_path = data_dir / "naip_train_buildings.geojson"
test_raster_path = data_dir / "naip_test.tif"

# Check if data exists, if not download it
if not train_raster_path.exists():
    print("Downloading datasets...")
    train_raster_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_rgb_train.tif"
    train_vector_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_train_buildings.geojson"
    test_raster_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_test.tif"
    
    train_raster_path = geoai.download_file(train_raster_url, output_path=str(train_raster_path))
    train_vector_path = geoai.download_file(train_vector_url, output_path=str(train_vector_path))
    test_raster_path = geoai.download_file(test_raster_url, output_path=str(test_raster_path))
    print("Data downloaded successfully!")
else:
    print("Using existing data from notebook 03")
    
print(f"\nData paths:")
print(f"  Training raster: {train_raster_path}")
print(f"  Training vectors: {train_vector_path}")
print(f"  Test raster: {test_raster_path}")

Using existing data from notebook 03

Data paths:
  Training raster: ../data/naip_rgb_train.tif
  Training vectors: ../data/naip_train_buildings.geojson
  Test raster: ../data/naip_test.tif


## 2. Coordinate Reference System (CRS) Alignment

**Key Concept from Notebook 01**: All geospatial datasets must share the same CRS to align properly. We will inspect each dataset and reproject as needed.

The test raster is in **EPSG:26911** (UTM Zone 11N). We need to ensure all vector data matches this CRS.

In [3]:
# Inspect raster CRS
with rasterio.open(test_raster_path) as src:
    raster_crs = src.crs
    raster_bounds = src.bounds
    print(f"Test Raster CRS: {raster_crs}")
    print(f"Test Raster Bounds: {raster_bounds}")
    print(f"Resolution: {src.res}")

# Load and inspect building footprints
buildings_gdf = gpd.read_file(train_vector_path)
print(f"\nBuildings CRS: {buildings_gdf.crs}")
print(f"Number of buildings: {len(buildings_gdf)}")
print(f"Buildings bounds:")
print(f"  {buildings_gdf.total_bounds}")

Test Raster CRS: EPSG:26911
Test Raster Bounds: BoundingBox(left=454641.6, bottom=5276774.4, right=456670.8, top=5277582.6)
Resolution: (0.6000000000000034, 0.5999999999994469)

Buildings CRS: EPSG:4326
Number of buildings: 735
Buildings bounds:
  [-117.6017984    47.65016239 -117.58246913   47.655846  ]


In [7]:
# Create a synthetic river/creek feature for demonstration
# In a real scenario, this would come from a hydrology dataset

# Get the extent of our test area
with rasterio.open(test_raster_path) as src:
    bounds = src.bounds
    center_x = (bounds.left + bounds.right) / 2
    center_y = (bounds.bottom + bounds.top) / 2
    
# Create a meandering river line within the test area
# Using the raster CRS (EPSG:26911 - meters)
river_coords = [
    (bounds.left + 50, center_y - 100),
    (center_x - 100, center_y - 50),
    (center_x, center_y),
    (center_x + 100, center_y + 50),
    (bounds.right - 50, center_y + 100)
]

river_line = LineString(river_coords)
river_gdf = gpd.GeoDataFrame(
    {"name": ["Creek"], "type": ["waterway"]},
    geometry=[river_line],
    crs=raster_crs  # Match the raster CRS
)

print(f"Created river feature:")
print(f"  CRS: {river_gdf.crs}")
print(f"  Length: {river_line.length:.1f} meters")

# Reproject buildings to match raster CRS if needed
if buildings_gdf.crs != raster_crs:
    print(f"\nReprojecting buildings from {buildings_gdf.crs} to {raster_crs}")
    buildings_gdf = buildings_gdf.to_crs(raster_crs)
    
print(f"\nAll datasets now aligned to: {raster_crs}")

Created river feature:
  CRS: PROJCS["NAD83 / UTM zone 11N",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-117],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","26911"]]
  Length: 1955.7 meters

All datasets now aligned to: EPSG:26911


## 3. Spatial Operations: Creating 6m Buffer Zones

**Key Concept from Notebook 01**: Buffer operations create zones around geometries. We will create a **6-meter buffer** around the river to identify flood risk zones.

This is critical for flood management - buildings within this buffer are at higher risk during flood events.

In [11]:
# Create 6m buffer around the river (flood risk zone)
buffer_distance = 6  # meters

river_buffer = river_line.buffer(buffer_distance)
buffer_gdf = gpd.GeoDataFrame(
    {"name": ["Flood Risk Zone"], "buffer_m": [buffer_distance]},
    geometry=[river_buffer],
    crs=raster_crs
)

print(f"Created {buffer_distance}m buffer around river")
print(f"Buffer area: {river_buffer.area:.1f} sq meters")
print(f"Buffer perimeter: {river_buffer.length:.1f} meters")

# Visualize the buffer
m = leafmap.Map()
m.add_raster(str(test_raster_path), bands=[1, 2, 3], layer_name="Test Imagery")
m.add_geojson(river_gdf, layer_name="River", style={"color": "blue", "weight": 3})
m.add_geojson(buffer_gdf, layer_name="6m Buffer Zone", 
              style={"color": "red", "fillOpacity": 0.3, "weight": 2})
m

Created 6m buffer around river
Buffer area: 23581.0 sq meters
Buffer perimeter: 3949.0 meters


Map(center=[47.6464835, -117.59043650000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom…

## 4. Machine Learning: Building Detection

**Key Concept from Notebook 03**: We will use semantic segmentation to automatically detect buildings from aerial imagery. This includes:

1. Preparing training data (tiles)
2. Training a U-Net model
3. Running inference on test imagery
4. Vectorizing predictions

Note: For this demonstration, we will train with reduced epochs. In production, you would train longer for better accuracy.

In [12]:
# Prepare training data tiles
print("Preparing training tiles...")

out_folder = str(data_dir / "encroachment_analysis")

tiles = geoai.export_geotiff_tiles(
    in_raster=str(train_raster_path),
    out_folder=out_folder,
    in_class_data=str(train_vector_path),
    tile_size=512,
    stride=256,
    buffer_radius=0,
)

print(f"Created {len(tiles)} training tiles")

Preparing training tiles...

Raster info for ../data/naip_rgb_train.tif:
  CRS: EPSG:26911
  Dimensions: 2503 x 1126
  Resolution: (0.6000000000000046, 0.6)
  Bands: 3
  Bounds: BoundingBox(left=454780.8, bottom=5277567.0, right=456282.6, top=5278242.6)
Loaded 735 features from ../data/naip_train_buildings.geojson
Vector CRS: EPSG:4326
Reprojecting features from EPSG:4326 to EPSG:26911
Found 1 unique classes: ['building']


Generated: 36, With features: 36: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:06<00:00,  5.84it/s]


------- Export Summary -------
Total tiles exported: 36
Tiles with features: 36 (100.0%)
Average feature pixels per tile: 46795.0
Output saved to: ../data/encroachment_analysis

------- Georeference Verification -------
Created 5 training tiles


In [13]:
# Train the semantic segmentation model
# Using fewer epochs for faster demonstration
print("Training U-Net model...")
print("(This may take 5-10 minutes...)")

model_dir = f"{out_folder}/unet_models"

geoai.train_segmentation_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/labels",
    output_dir=model_dir,
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=3,
    num_classes=2,  # background + building
    batch_size=8,
    num_epochs=5,  # Reduced for demo - use 10+ for production
    learning_rate=0.001,
    val_split=0.2,
    verbose=True,
)

print(f"\nModel training complete! Saved to: {model_dir}")

Training U-Net model...
(This may take 5-10 minutes...)
Using device: cuda
Found 36 image files and 36 label files
Training on 28 images, validating on 8 images
Checking image sizes for compatibility...
All sampled images have the same size: (512, 512)
No resizing needed.
Testing data loader...
Data loader test passed.
Starting training with unet + resnet34
Model parameters: 24,436,514
Epoch: 1, Batch: 1/4, Loss: 0.5944, Time: 7.06s
Epoch 1/5: Train Loss: 0.4697, Val Loss: 0.4332, Val IoU: 0.4902, Val F1: 0.5835, Val Precision: 0.6575, Val Recall: 0.5762
Saving best model with IoU: 0.4902
Epoch: 2, Batch: 1/4, Loss: 0.3526, Time: 5.22s
Epoch 2/5: Train Loss: 0.3015, Val Loss: 1.3010, Val IoU: 0.5125, Val F1: 0.6136, Val Precision: 0.6918, Val Recall: 0.5977
Saving best model with IoU: 0.5125
Epoch: 3, Batch: 1/4, Loss: 0.2357, Time: 5.42s
Epoch 3/5: Train Loss: 0.2000, Val Loss: 8.7563, Val IoU: 0.4317, Val F1: 0.4876, Val Precision: 0.5797, Val Recall: 0.5122
Epoch: 4, Batch: 1/4, Los

In [14]:
# Run inference on test imagery
print("Running inference on test imagery...")

prediction_path = str(data_dir / "test_prediction.tif")
model_path = f"{model_dir}/best_model.pth"

geoai.semantic_segmentation(
    input_path=str(test_raster_path),
    output_path=prediction_path,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=3,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=4,
)

print(f"\nPrediction saved to: {prediction_path}")

Running inference on test imagery...
Input file format: GeoTIFF (.tif)
Processing 65 windows...


84it [00:03, 25.19it/s]                                                                                                                                          


Predicted classes: 2 classes, Background: 95.3%
Inference completed in 3.73 seconds
Saved prediction to ../data/test_prediction.tif

Prediction saved to: ../data/test_prediction.tif


In [15]:
# Vectorize predictions to get building polygons
print("Vectorizing predictions...")

detected_buildings_path = str(data_dir / "detected_buildings.geojson")

detected_buildings = geoai.orthogonalize(
    prediction_path, 
    detected_buildings_path, 
    epsilon=2
)

# Add geometric properties
detected_buildings = geoai.add_geometric_properties(
    detected_buildings, 
    area_unit="m2", 
    length_unit="m"
)

# Filter out small artifacts (likely false positives)
min_building_area = 10  # sq meters
detected_buildings = detected_buildings[detected_buildings["area_m2"] > min_building_area]

print(f"\nDetected {len(detected_buildings)} buildings (> {min_building_area} m²)")
print(f"\nBuilding area statistics:")
print(f"  Mean: {detected_buildings['area_m2'].mean():.1f} m²")
print(f"  Median: {detected_buildings['area_m2'].median():.1f} m²")
print(f"  Min: {detected_buildings['area_m2'].min():.1f} m²")
print(f"  Max: {detected_buildings['area_m2'].max():.1f} m²")

Vectorizing predictions...
Processing 2442 features...


Converting features: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 2442/2442 [00:02<00:00, 964.59shape/s]


Saving to ../data/detected_buildings.geojson...
Done!

Detected 320 buildings (> 10 m²)

Building area statistics:
  Mean: 238.7 m²
  Median: 70.6 m²
  Min: 10.1 m²
  Max: 4592.2 m²


## 5. Spatial Analysis: Finding At-Risk Buildings

**Key Concept from Notebooks 01 and 02**: We will perform a spatial join to identify buildings that intersect with the 6m flood risk buffer zone.

This is the critical analysis step that answers: **Which buildings are at risk?**

In [23]:
# Spatial join: Find buildings within the buffer zone
print("Performing spatial analysis...")

# Ensure detected buildings have the same CRS
if detected_buildings.crs != buffer_gdf.crs:
    detected_buildings = detected_buildings.to_crs(buffer_gdf.crs)

# Method 1: Using spatial join
at_risk_buildings = gpd.sjoin(
    detected_buildings, 
    buffer_gdf, 
    how="inner", 
    predicate="intersects"
)

print(f"\nSpatial Analysis Results:")
print(f"  Total buildings detected: {len(detected_buildings)}")
print(f"  Buildings in flood risk zone: {len(at_risk_buildings)}")
print(f"  Percentage at risk: {len(at_risk_buildings)/len(detected_buildings)*100:.1f}%")

# Calculate total area at risk
total_at_risk_area = at_risk_buildings["area_m2"].sum()
total_building_area = detected_buildings["area_m2"].sum()

print(f"\nArea Analysis:")
print(f"  Total building area: {total_building_area:.1f} m²")
print(f"  At-risk building area: {total_at_risk_area:.1f} m²")
print(f"  Percentage area at risk: {total_at_risk_area/total_building_area*100:.1f}%")

Performing spatial analysis...

Spatial Analysis Results:
  Total buildings detected: 320
  Buildings in flood risk zone: 16
  Percentage at risk: 5.0%

Area Analysis:
  Total building area: 76384.1 m²
  At-risk building area: 8683.2 m²
  Percentage area at risk: 11.4%


In [24]:
# Calculate distance from each building to the river
# (Additional analysis for prioritization)

detected_buildings["distance_to_river_m"] = detected_buildings.geometry.distance(river_line)

# Categorize risk levels
def categorize_risk(distance):
    if distance <= 6:
        return "High Risk (<=6m)"
    elif distance <= 15:
        return "Medium Risk (7-15m)"
    else:
        return "Low Risk (>15m)"

detected_buildings["risk_category"] = detected_buildings["distance_to_river_m"].apply(categorize_risk)

# Summary by risk category
print("\nRisk Category Summary:")
risk_summary = detected_buildings.groupby("risk_category").agg({
    "area_m2": ["count", "sum", "mean"]
}).round(1)
print(risk_summary)


Risk Category Summary:
                    area_m2                
                      count      sum   mean
risk_category                              
High Risk (<=6m)         16   8683.2  542.7
Low Risk (>15m)         287  64628.3  225.2
Medium Risk (7-15m)      17   3072.6  180.7


## 6. Comprehensive Visualization

**Key Concept from Notebook 02**: Create an interactive map showing all layers:
- Satellite imagery (raster)
- River and buffer zone (vector)
- All detected buildings (vector)
- At-risk buildings highlighted (vector)

In [25]:
# Create comprehensive interactive map
m = leafmap.Map()

# Add satellite imagery as base
m.add_raster(str(test_raster_path), bands=[1, 2, 3], layer_name="NAIP Imagery")

# Add river
m.add_geojson(
    river_gdf, 
    layer_name="River/Creek", 
    style={"color": "blue", "weight": 4}
)

# Add buffer zone
m.add_geojson(
    buffer_gdf, 
    layer_name="Flood Risk Zone (6m)", 
    style={"color": "orange", "fillOpacity": 0.2, "weight": 2}
)

# Add all buildings (color by risk category)
m.add_geojson(
    detected_buildings, 
    layer_name="All Buildings",
    style={"fillOpacity": 0.6, "weight": 1},
    fill_colors=["#ff0000", "#ffaa00", "#00aa00"],
)

# Add at-risk buildings with emphasis
if len(at_risk_buildings) > 0:
    
    m.add_gdf(
        at_risk_buildings,
        layer_name="At-Risk Buildings",
        style={"color": "red", "fillOpacity": 0.8, "weight": 2}
    )

m

Map(center=[47.6464835, -117.59043650000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom…

## 7. Results Export and Reporting

Export the analysis results for stakeholders and generate a summary report.

In [27]:
# Export at-risk buildings
output_path = str(data_dir / "at_risk_buildings.geojson")
at_risk_export = at_risk_buildings[["geometry", "area_m2", "perimeter_m", "distance_to_river_m"]]
at_risk_export.to_file(output_path, driver="GeoJSON")

print(f"\nExported at-risk buildings to: {output_path}")

# Generate summary report
report_lines = []
report_lines.append("===============================================")
report_lines.append("RIVER ENCROACHMENT ANALYSIS REPORT")
report_lines.append("===============================================")
report_lines.append("")
report_lines.append("ANALYSIS PARAMETERS")
report_lines.append("-------------------")
report_lines.append("Buffer Distance: 6 meters")
report_lines.append("Study Area: Test raster imagery")
report_lines.append(f"CRS: {raster_crs}")
report_lines.append("")
report_lines.append("BUILDING DETECTION RESULTS")
report_lines.append("--------------------------")
report_lines.append(f"Total Buildings Detected: {len(detected_buildings)}")
report_lines.append(f"Total Building Area: {total_building_area:.1f} m²")
report_lines.append(f"Average Building Size: {detected_buildings['area_m2'].mean():.1f} m²")
report_lines.append("")
report_lines.append("RISK ASSESSMENT")
report_lines.append("---------------")
report_lines.append(f"Buildings in Flood Risk Zone (<=6m): {len(at_risk_buildings)}")
report_lines.append(f"Percentage of Buildings at Risk: {len(at_risk_buildings)/len(detected_buildings)*100:.1f}%")
report_lines.append(f"Total Area at Risk: {total_at_risk_area:.1f} m²")
report_lines.append("")
report_lines.append("RISK BREAKDOWN")
report_lines.append("--------------")

for category, count in detected_buildings["risk_category"].value_counts().items():
    pct = count / len(detected_buildings) * 100
    report_lines.append(f"  {category}: {count} buildings ({pct:.1f}%)")

report_lines.append("")
report_lines.append("RECOMMENDATIONS")
report_lines.append("---------------")
report_lines.append(f"1. Prioritize inspection of {len(at_risk_buildings)} buildings in the 6m buffer zone")
report_lines.append("2. Consider flood mitigation measures for high-risk structures")
report_lines.append("3. Monitor medium-risk buildings (7-15m) during heavy rainfall events")
report_lines.append("4. Update building inventory with ground-truth verification")
report_lines.append("")
report_lines.append("OUTPUT FILES")
report_lines.append("------------")
report_lines.append("- At-risk buildings: at_risk_buildings.geojson")
report_lines.append("- All detected buildings: detected_buildings.geojson")
report_lines.append("- Prediction raster: test_prediction.tif")
report_lines.append("")
report_lines.append("===============================================")

report = "\n".join(report_lines)

# Save report
report_path = str(data_dir / "encroachment_report.txt")
with open(report_path, "w") as f:
    f.write(report)

print(report)
print(f"\nReport saved to: {report_path}")


Exported at-risk buildings to: ../data/at_risk_buildings.geojson
RIVER ENCROACHMENT ANALYSIS REPORT

ANALYSIS PARAMETERS
-------------------
Buffer Distance: 6 meters
Study Area: Test raster imagery
CRS: EPSG:26911

BUILDING DETECTION RESULTS
--------------------------
Total Buildings Detected: 320
Total Building Area: 76384.1 m²
Average Building Size: 238.7 m²

RISK ASSESSMENT
---------------
Buildings in Flood Risk Zone (<=6m): 16
Percentage of Buildings at Risk: 5.0%
Total Area at Risk: 8683.2 m²

RISK BREAKDOWN
--------------
  Low Risk (>15m): 287 buildings (89.7%)
  Medium Risk (7-15m): 17 buildings (5.3%)
  High Risk (<=6m): 16 buildings (5.0%)

RECOMMENDATIONS
---------------
1. Prioritize inspection of 16 buildings in the 6m buffer zone
2. Consider flood mitigation measures for high-risk structures
3. Monitor medium-risk buildings (7-15m) during heavy rainfall events
4. Update building inventory with ground-truth verification

OUTPUT FILES
------------
- At-risk buildings: at

## 8. Summary: Complete Workflow Recap

This project demonstrated a complete geospatial ML workflow:

| Step | Concept | Tools Used | Key Takeaway |
|------|---------|------------|--------------|
| **1. Setup** | Data Loading | `geoai`, `geopandas` | Ensure data availability and consistency |
| **2. CRS Alignment** | Coordinate Systems | `rasterio`, `to_crs()` | All datasets must share the same CRS |
| **3. Buffer Zones** | Spatial Operations | `shapely.buffer()` | Create risk zones around features |
| **4. ML Detection** | Semantic Segmentation | `geoai` U-Net | Automate feature extraction from imagery |
| **5. Spatial Join** | Spatial Analysis | `gpd.sjoin()` | Find intersecting features |
| **6. Visualization** | Interactive Maps | `leafmap` | Communicate results effectively |
| **7. Export** | Data Sharing | `GeoJSON`, reports | Enable decision-making |

### Key Learnings

1. **Data Integration**: Raster and vector data require CRS alignment before analysis
2. **Spatial Operations**: Buffer zones are essential for proximity-based risk assessment
3. **Machine Learning**: U-Net semantic segmentation automates building detection from aerial imagery
4. **Spatial Analysis**: Spatial joins enable complex queries like "buildings near rivers"
5. **Visualization**: Interactive maps communicate complex spatial relationships clearly

### Real-World Applications

This workflow applies to:
- **Flood Risk Assessment**: Identifying structures in flood plains
- **Urban Planning**: Zoning and development control near waterways
- **Emergency Response**: Prioritizing evacuations during flood events
- **Insurance**: Risk-based pricing for flood insurance
- **Environmental Monitoring**: Tracking encroachment on natural waterways

### Best Practices

- Always verify CRS before spatial operations
- Use projected CRS (meters) for distance calculations
- Filter ML predictions to remove small artifacts
- Visualize results with context (basemaps, legends)
- Document analysis parameters for reproducibility
- Export results in standard formats (GeoJSON, Shapefile)

---

**Congratulations!** You have completed the end-to-end geospatial ML workflow from data loading to actionable insights.

In [ ]:
# Optional: Clean up large temporary files to save disk space
# Uncomment if needed:

# import shutil
# shutil.rmtree(out_folder, ignore_errors=True)
# print("Cleaned up temporary training files")

print("\nAnalysis complete!")
print(f"\nGenerated files in {data_dir}:")
for f in data_dir.glob("*.geojson"):
    print(f"  - {f.name}")
for f in data_dir.glob("*.txt"):
    print(f"  - {f.name}")
for f in data_dir.glob("*prediction*.tif"):
    print(f"  - {f.name}")